In [ ]:
'''
O que este pipeline faz

Resample de alta qualidade de todos os áudios para 16kHz (usando kaiser_best)
Data augmentation específico para os 8 fonemas raros (triplicando estas amostras)
Oversampling moderado para idade 8-11 (triplicando, mas com augmentation suave)
Salva tudo em formato otimizado (WAV + CSV de metadados)
Previne data leakage no split treino/validação
O resultado final será um dataset balanceado e pronto para fine-tuning do WavLM! 
'''

In [1]:
import os

# Mudar para o diretório correto onde estão seus arquivos
caminho_correto = "/Users/daltonlintz/Desktop/Competicoes/DD_MS_-_Childrens_Speech_Recognition_Challenge_-_Phonetic_Track/childrens-speech-recognition-runtime/notebooks"
os.chdir(caminho_correto)

# Verificar se funcionou
print(f"✅ Diretório atual alterado para: {os.getcwd()}")
print(f"✅ Arquivos .py agora: {[f for f in os.listdir('.') if f.endswith('.py')]}")

from preprocessing_pipeline import PreprocessingPipeline

pipeline = PreprocessingPipeline()
df_metadata = pipeline.run()

✅ Diretório atual alterado para: /Users/daltonlintz/Desktop/Competicoes/DD_MS_-_Childrens_Speech_Recognition_Challenge_-_Phonetic_Track/childrens-speech-recognition-runtime/notebooks
✅ Arquivos .py agora: ['preprocessing_pipeline.py']
✅ resampy instalado
✅ soundfile instalado
📁 Diretórios criados/verificados:
   • ../data/processed
   • ../data/processed/audio

📂 Carregando transcrições...
   ✅ Total: 12043 amostras

🔍 Identificando amostras para augmentation...


Analisando fonemas raros: 100%|██████████| 12043/12043 [00:00<00:00, 72444.61it/s]


   • 38 amostras idade 8-11
   • 110 amostras com fonemas raros

🚀 INICIANDO PIPELINE DE PRÉ-PROCESSAMENTO

🔄 Processando áudios base...


Processando: 100%|██████████| 12043/12043 [05:18<00:00, 37.83it/s]



✅ 12043 áudios base processados com sucesso

🔄 Gerando versões aumentadas...


Idade rara: 100%|██████████| 38/38 [00:08<00:00,  4.22it/s]



📊 Criando metadados finais...

✅ PIPELINE CONCLUÍDO!

📈 ESTATÍSTICAS FINAIS:
   • Amostras originais: 12043
   • Amostras após pipeline: 14809
   • Fator de aumento: 1.2x
   
   • Amostras aumentadas: 2766
   • Amostras não aumentadas: 12043
   
📁 Arquivos salvos:
   • Áudios: ../data/processed/audio/
   • Metadados: ../data/processed/metadata.csv
        


In [2]:
# ============================================
# VERIFICAR DATASET PROCESSADO
# ============================================

import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# Carregar metadados
metadata_path = Path("../data/processed/metadata.csv")
df = pd.read_csv(metadata_path)

print("="*60)
print("📊 DATASET PROCESSADO - ESTATÍSTICAS")
print("="*60)

print(f"\n📍 Total de amostras: {len(df)}")
print(f"📍 Amostras originais: {len(df[df['augmented'] == False])}")
print(f"📍 Amostras aumentadas: {len(df[df['augmented'] == True])}")

print("\n📈 Distribuição por idade:")
print(df['age_bucket'].value_counts().sort_index())

print("\n🔄 Tipos de augmentation:")
print(df['augmentation_type'].value_counts())

print("\n🔤 Exemplos de transcrições:")
for i, row in df.sample(5).iterrows():
    print(f"\n  ID: {row['id']}")
    print(f"  Transcrição: {row['phonetic_text']}")
    print(f"  Idade: {row['age_bucket']}")
    print(f"  Aumentada: {row['augmented']} ({row['augmentation_type']})")

# ============================================
# CRIAR SPLIT TREINO/VALIDAÇÃO
# ============================================

from sklearn.model_selection import train_test_split

# Usar IDs originais para evitar data leakage
unique_ids = df['original_id'].unique()

# Estratificar por idade
idade_por_id = df.groupby('original_id')['age_bucket'].first()
stratify = idade_por_id.loc[unique_ids].values

# Split 90/10
ids_train, ids_val = train_test_split(
    unique_ids, 
    test_size=0.1, 
    random_state=42,
    stratify=stratify
)

train_df = df[df['original_id'].isin(ids_train)]
val_df = df[df['original_id'].isin(ids_val)]

print("\n" + "="*60)
print("🎯 SPLIT TREINO/VALIDAÇÃO")
print("="*60)
print(f"\n📚 TREINO:")
print(f"   • Total amostras: {len(train_df)}")
print(f"   • IDs originais: {len(ids_train)}")
print(f"   • Distribuição idade:")
print(train_df['age_bucket'].value_counts().sort_index())

print(f"\n🔬 VALIDAÇÃO:")
print(f"   • Total amostras: {len(val_df)}")
print(f"   • IDs originais: {len(ids_val)}")
print(f"   • Distribuição idade:")
print(val_df['age_bucket'].value_counts().sort_index())

# ============================================
# SALVAR SPLITS
# ============================================

train_df.to_csv("../data/processed/train_split.csv", index=False)
val_df.to_csv("../data/processed/val_split.csv", index=False)

print("\n✅ Splits salvos em: ../data/processed/")

📊 DATASET PROCESSADO - ESTATÍSTICAS

📍 Total de amostras: 14809
📍 Amostras originais: 12043
📍 Amostras aumentadas: 2766

📈 Distribuição por idade:
age_bucket
3-4     11771
5-7      2544
8-11      494
Name: count, dtype: int64

🔄 Tipos de augmentation:
augmentation_type
none         12043
pitch_-2       444
pitch_-1       444
pitch_1        444
pitch_2        444
speed_0.9      330
speed_1.1      330
noise          330
Name: count, dtype: int64

🔤 Exemplos de transcrições:

  ID: U_c8c81c545ce3d1dd
  Transcrição: wi hæd tu stɑp plein nsɑid bikʌz ɪt hæz wein soʊ mʌʃ
  Idade: 5-7
  Aumentada: False (none)

  ID: U_52e9a75a45018795
  Transcrição: tɛl
  Idade: 5-7
  Aumentada: False (none)

  ID: U_74022d471bb122be
  Transcrição: dəd ɪt god heɹe fɪtθ nop θkwið nop dəd ɪt go hijʊ do jæ
  Idade: 3-4
  Aumentada: False (none)

  ID: U_c97f4c0fde0f4de8_speed_0.9
  Transcrição: ə jæ æ mɑi ʃwɛ hæː tɔiç çɑik ei kɑɹ
  Idade: 3-4
  Aumentada: True (speed_0.9)

  ID: U_0afe7581d6580718
  Transcrição: